# Python to SQL, and back again
In this codealong we will show you how to create a relational database from your pandas DataFrames.
> **To run this notebook you will need to work locally and not on colab.**

---
## 1.&nbsp; Import libraries 💾
If you haven't already installed sqlalchemy, you will need to. Uncomment the code below, install, and then recomment the code - you only need to install it once.

In [ ]:
# install if needed
#!pip install sqlalchemy
#!pip install pandas
#!pip install pymysql

In [1]:
import pandas as pd

---
## 2.&nbsp; Relational Databases 📂

Creating DataFrames in python and pandas often results in tables with repeated information, as shown in the example below.
<br>

| author_name | book_title | year_published |
| --- | --- | --- |
| Arthur Conan Doyle | The Adventures of Sherlock Holmes | 1887 |
| J.R.R. Tolkien | The Hobbit | 1937 |
| J.R.R. Tolkien | The Lord of the Rings | 1954 |
| Harper Lee | To Kill a Mockingbird | 1960 |
| Harper Lee | Go Set a Watchman | 2015 |
<br>

This can be problematic for relational databases, which are designed to store data efficiently and avoid redundancy. To address this issue, we will separate the author and book information into two tables: authors and books. This approach eliminates duplicate data, ensuring data integrity and optimising storage.
<br>

| author_id | author_name |
| --- | --- |
| 1 | Arthur Conan Doyle |
| 2 | J.R.R. Tolkien |
| 3 | Harper Lee |
<br>

| book_id | book_title | year_published | author_id |
|---|---|---|---|
| 1 | The Adventures of Sherlock Holmes | 1887 | 1 |
| 2 | The Hobbit | 1937 | 2 |
| 3 | The Lord of the Rings | 1954 | 2 |
| 4 | To Kill a Mockingbird | 1960 | 3 |
| 5 | Go Set a Watchman | 2015 | 3 |

---
## 3.&nbsp; Creating the authors table with python 🐍
Let's start by creating the original DataFrame, including the repeated data.

In [ ]:
names = ["Arthur Conan Doyle", "J.R.R. Tolkien", "J.R.R. Tolkien", "Harper Lee", "Harper Lee"]
titles = ["The Adventures of Sherlock Holmes", "The Hobbit", "The Lord of the Rings", "To Kill a Mockingbird", "Go Set a Watchman"]
years = [1887, 1937, 1954, 1960, 2015]

non_relational_df = pd.DataFrame({"author_name": names,
                                  "book_title": titles,
                                  "year_published": years})

non_relational_df

Now, let's select only the authors without any duplicates.

In [ ]:
authors_unique = non_relational_df["author_name"].unique()

authors_df = pd.DataFrame({"author_name": authors_unique})

authors_df

Fantastic! This DataFrame will be the foundation of our authors table.

---
## 4.&nbsp; Creating the matching authors table with SQL 💻

Ok, now we're ready to store this DataFrame in SQL. Before we can send the information in SQL, we need to make a table that has the same columns and data types to recieve the data. While we are creating a table for authors, we can also create the books table too.

Open MySQL Workbench, open a local connection, and open a new file. Then copy and paste the code from below.

```sql
-- Drop the database if it already exists
DROP DATABASE IF EXISTS sql_workshop ;

-- Create the database
CREATE DATABASE sql_workshop;

-- Use the database
USE sql_workshop;

-- Create the 'authors' table
CREATE TABLE authors (
    author_id INT AUTO_INCREMENT, -- Automatically generated ID for each author
    author_name VARCHAR(255) NOT NULL, -- Name of the author
    PRIMARY KEY (author_id) -- Primary key to uniquely identify each author
);

-- Create the 'books' table
CREATE TABLE books (
    book_id INT AUTO_INCREMENT, -- Automatically generated ID for each book
    book_title VARCHAR(255) NOT NULL, -- Title of the book
    year_published INT, -- Year the book was published
    author_id INT, -- ID of the author who wrote the book
    PRIMARY KEY (book_id), -- Primary key to uniquely identify each book
    FOREIGN KEY (author_id) REFERENCES authors(author_id) -- Foreign key to connect each book to its author
);
```

To download the sql file that we will follow for this section, [click here](https://drive.google.com/uc?export=download&id=1tln_33FM7D9wLckzxacBJNcMYqtyxybE)

If you'd like more information about MySQL data types [click here](https://www.w3schools.com/mysql/mysql_datatypes.asp).

---
## 5.&nbsp; Sending the information from this notebook to sql 📠
To establish a connection with the SQL database, we need to provide the notebook with the necessary information, which we do using the connection string below. You will need to modify only the password variable, which should match the password you set during MySQL Workbench installation.

In [2]:
pip install dotenv


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#!pip install python-dotenv
from dotenv import load_dotenv
import os
load_dotenv()  # Load .env into environment

True

In [3]:

from sqlalchemy import create_engine
schema = "sql_workshop"
host = "127.0.0.1"
user = "root"
password = os.getenv("MYSQL_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
engine = create_engine(connection_string)
print("Connection successful!")

Connection successful!


To send information to our sql databse we use the pandas method `.to_sql()`. The argument `if_exists="append"` says that we don't want to overwrite any existing data, but add on to what is already there.

In [ ]:
authors_df.to_sql('authors',
                  if_exists='append',
                  con=connection_string,
                  index=False)

Now, have a look at the table `authors` in MySQL Workbench, you should see that the names of the authors have appeared.

---
## 6.&nbsp; Retrieving information from sql to this notebook 📥
It's not only possible to send information to a SQL database, but also retrieve it too. Using `.read_sql()` in combination with the `connection_string` we can access the required data.

In [ ]:
authors_from_sql = pd.read_sql("authors", con=connection_string)
authors_from_sql

Using this same method, we can also perform SQL queries to only bring back certain sections of information instead of the whole DataFrame.

In [ ]:
pd.read_sql("""
            SELECT DISTINCT author_name
            FROM authors
            """,
            con=connection_string)

---
## 7.&nbsp; Preparing and sending the books table 📚
By extracting the authors table from our SQL database, we gain access to the unique identifier `author_id` assigned to each author. These `author_id`'s serve as pointers to their corresponding author records, allowing us to seamlessly link the `author_id`'s in the books table to their respective authors in the authors table, thereby completing the books table.

In [ ]:
books_df = non_relational_df.merge(authors_from_sql,
                                   on = "author_name",
                                   how="left")

books_df

In [ ]:
books_df = books_df.drop(columns=["author_name"])

books_df

In [ ]:
books_df.to_sql('books',
                if_exists='append',
                con=connection_string,
                index=False)

In [ ]:
books_from_sql = pd.read_sql("books", con=connection_string)
books_from_sql

---
## 8.&nbsp; Challenge 😃
Now that you've learnt how to send and retrieve information, it's your turn to show off your skills. Create multiple tables in SQL for the data you scrapped about cities from Wikipedia. One should just be a table about the cities, the others should be facts about the cities.

| city_id | city |
| --- | --- |
| 1 | Berlin |
| 2 | Hamburg |
| 3 | Munich |

<br>

| City ID | Population | Year Data Retrieved |
|---|---|---|
| 1 | 3,850,809 | 2024 |
| 2 | 1,945,532 | 2024 |
| 3 | 1,512,491 | 2024 |

> **Pro Tip:** Visualise your relational database with pen and paper before you start coding. This can help you to identify any potential problems or inconsistencies in your design, and it can also make the coding process more efficient.

### Cities DF

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from lat_lon_parser import parse    # for decimal coordinates


def cities_dataframe(cities):

  city_data = []

  for city in cities:
    url = f"https://www.wikipedia.org/wiki/{city}"
    headers = {'User-Agent': 'Chrome/134.0.0.0'}

    response = requests.get(url, headers=headers)
    city_soup = BeautifulSoup(response.content, 'html.parser')

    # extract the relevant information
    city_latitude = city_soup.find(class_="latitude").get_text()
    city_longitude = city_soup.find(class_="longitude").get_text()
    country = city_soup.find(class_="infobox-data").get_text()

    # keep track of data per city
    city_data.append({"city": city,
                    "country": country,
                    "latitude": parse(city_latitude), # latitude in decimal format
                    "longitude": parse(city_longitude), # longitude in decimal format
                    })

  return pd.DataFrame(city_data)

In [2]:
# call the function
list_of_cities = ["Berlin", "Hamburg", "Munich"]

cities_df = cities_dataframe(list_of_cities)
cities_df

,city,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


In [3]:
cities_df.dtypes

city          object
country       object
latitude     float64
longitude    float64
dtype: object

### Pop DF


In [4]:
from datetime import datetime # to get today's date

def populations_dataframe(cities):

    population_data = []

    for city in cities:
        url = f"https://www.wikipedia.org/wiki/{city}"
        headers = {'User-Agent': 'Chrome/134.0.0.0'}

        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, 'html.parser')

        # extract the relevant information
        city_population = city_soup.find(string="Population").find_next("td").get_text()
        city_population_clean = int(city_population.replace(",", ""))
        today = datetime.today().strftime("%d.%m.%Y")
        today = pd.to_datetime(today, format="%d.%m.%Y")

        # keep track of data per city
        population_data.append({"city": city,
                        "population": city_population_clean,
                        "timestamp_population": today
                        })

    return pd.DataFrame(population_data)

In [5]:
# call the populations function
cities = ["Berlin", "Hamburg", "Munich"]

population_df = populations_dataframe(cities)
population_df

,city,population,timestamp_population
0,Berlin,3596999,2026-05-27
1,Hamburg,1973896,2026-05-27
2,Munich,1505005,2026-05-27


In [6]:
population_df.dtypes


city                            object
population                       int64
timestamp_population    datetime64[ns]
dtype: object

### Send to SQL using


```sql
-- Drop the database if it already exists
DROP DATABASE IF EXISTS sql_workshop_cloud;

-- Create the database
CREATE DATABASE sql_workshop_cloud;

-- Use the database
USE sql_workshop_cloud;

-- Create the 'cities' table
CREATE TABLE cities (
    city_id INT AUTO_INCREMENT,             -- Automatically generated ID for each city
    city VARCHAR(255) NOT NULL,  
    country VARCHAR(255) NOT NULL,
    latitude FLOAT NOT NULL,
    longitude FLOAT NOT NULL,
    PRIMARY KEY (city_id)                   -- Primary key to uniquely identify each city
);

SELECT * FROM cities;


```

In [7]:
# Sending the information from this notebook to sql
from dotenv import load_dotenv
import os
load_dotenv()  # Load .env into environment

schema = "gans_db"
host = "127.0.0.1"
user = "root"
password = os.getenv("MYSQL_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [15]:
# Step 1: Insert cities_df into MySQL
cities_df.to_sql(name='cities', con=connection_string, if_exists='append', index=False)


3

In [16]:
# Step 2: Retrieve the cities with their auto-generated IDs
cities_in_db = pd.read_sql("SELECT city_id, city FROM cities", con=connection_string)
cities_in_db

,city_id,city
0,1,Berlin
1,2,Hamburg
2,3,Munich


In [17]:
# Step 3: Merge population_df with city IDs
merged_population_df = population_df.merge(cities_in_db, on="city", how="left")
merged_population_df


,city,population,timestamp_population,city_id
0,Berlin,3596999,2026-05-27,1
1,Hamburg,1973896,2026-05-27,2
2,Munich,1505005,2026-05-27,3


In [18]:
# Step 4: Prepare final DataFrame
final_population_df = merged_population_df[['city_id', 'population', 'timestamp_population']]

### Send to SQL

```sql
-- Create the 'population' table
CREATE TABLE population (
    city_id INT NOT NULL,
    population INT NOT NULL,                          -- Population
    timestamp_population DATE NOT NULL,               -- Year or full date as string
    PRIMARY KEY (city_id, timestamp_population),      -- Composite primary key
    FOREIGN KEY (city_id) REFERENCES cities(city_id)  -- Foreign key constraint
);
```

In [ ]:
# Step 5: Insert into the population table
final_population_df.to_sql('population', con=connection_string, if_exists='append', index=False)


3

## Everything in one function(Bonus)


In [20]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from sqlalchemy.exc import IntegrityError
from lat_lon_parser import parse
from dotenv import load_dotenv
import os

load_dotenv()  # Load .env into environment

schema = "gans_db"
host = "127.0.0.1"
user = "root"
password = os.getenv("MYSQL_PASSWORD") # or just manually input your password as a string
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
# Assuming connection to MySQL database using SQLAlchemy
engine = create_engine(connection_string)


def populations_dataframe(cities):
    """Scrapes population data for cities and returns it as a DataFrame."""
    population_data = []

    for city in cities:
        url = f"https://www.wikipedia.org/wiki/{city}"
        headers = {'User-Agent': 'Chrome/134.0.0.0'}

        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, 'html.parser')

        # Extract the population data
        city_population = city_soup.find(string="Population").find_next("td").get_text()
        city_population_clean = int(city_population.replace(",", ""))
        today = datetime.today().strftime("%d.%m.%Y")
        today = pd.to_datetime(today, format="%d.%m.%Y")

        population_data.append({"city": city,
                                "population": city_population_clean,
                                "timestamp_population": today})

    return pd.DataFrame(population_data)

def cities_dataframe(cities):
    """Scrapes city details such as Country, Latitude, Longitude and returns as a DataFrame."""
    city_data = []

    for city in cities:
        url = f"https://www.wikipedia.org/wiki/{city}"
        headers = {'User-Agent': 'Chrome/134.0.0.0'}

        response = requests.get(url, headers=headers)
        city_soup = BeautifulSoup(response.content, 'html.parser')

        # Extract city details
        city_latitude = city_soup.find(class_="latitude").get_text()
        city_longitude = city_soup.find(class_="longitude").get_text()
        country = city_soup.find(class_="infobox-data").get_text()

        city_data.append({"city": city,
                          "country": country,
                          "latitude": parse(city_latitude),  # latitude in decimal format
                          "longitude": parse(city_longitude),  # longitude in decimal format
                          })

    return pd.DataFrame(city_data)

from sqlalchemy import text

def update_sql_database(cities_df, population_df):
    """Function to update cities and population tables in the database."""

    with engine.begin() as connection:  # begin() auto-commits at the end
        # Insert or update the cities table
        for _, row in cities_df.iterrows():
            try:
                connection.execute(
                    text("""
                        INSERT INTO cities (city, country, latitude, longitude)
                        VALUES (:city, :country, :latitude, :longitude)
                        ON DUPLICATE KEY UPDATE
                            city = VALUES(city),
                            country = VALUES(country),
                            latitude = VALUES(latitude),
                            longitude = VALUES(longitude)
                    """),
                    {
                        "city": row['city'],
                        "country": row['country'],
                        "latitude": row['latitude'],
                        "longitude": row['longitude']
                    }
                )
            except IntegrityError:
                print(f"City '{row['city']}' already exists.")

        # Insert or update the population table
        for _, row in population_df.iterrows():
            try:
                # First, get the city_id based on the city name
                result = connection.execute(
                    text("SELECT city_id FROM cities WHERE city = :city"),
                    {"city": row['city']}
                ).fetchone()

                if result:
                    city_id = result[0]

                    connection.execute(
                        text("""
                            INSERT INTO population (city_id, population, timestamp_population)
                            VALUES (:city_id, :population, :timestamp_population)
                            ON DUPLICATE KEY UPDATE
                                population = VALUES(population)
                        """),
                        {
                            "city_id": city_id,
                            "population": row['population'],
                            "timestamp_population": row['timestamp_population']
                        }
                    )
                else:
                    print(f"City '{row['city']}' not found in cities table.")

            except IntegrityError:
                print(f"Duplicate or error occurred for city '{row['city']}'.")


def main(cities):
    """Main function to combine both city and population scraping, then update the SQL tables."""

    # Get the dataframes
    cities_df = cities_dataframe(cities)
    population_df = populations_dataframe(cities)

    # Update the database
    update_sql_database(cities_df, population_df)

# Example usage
cities = ["Berlin", "Hamburg", "Munich"]  # Add any list of cities

main(cities)
